In [1]:
!pip install monai nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.2 MB/s  0:00:00


In [2]:
from google.colab import drive
import zipfile

drive.mount('/content/drive')
ZIP_FILE_PATH = '/content/drive/MyDrive/DS Project/Copy of bbdd_kidney.zip'
EXTRACT_DIR = '/content/dataset'
with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

Mounted at /content/drive


In [5]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
import scipy.io as sio

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix
import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = True

from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    ScaleIntensityRangePercentilesd,
    RandRotated,
    RandAdjustContrastd,
    RandGaussianNoised,
    Resized,
    EnsureTyped,
)


train_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim=-1),
    ScaleIntensityRangePercentilesd(
        keys=["image"], lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image"], spatial_size=(224, 224)),
    RandRotated(keys=["image"], range_x=0.26, prob=0.5, mode="bilinear"),
    RandAdjustContrastd(keys=["image"], prob=0.4, gamma=(0.8, 1.2)),
    RandGaussianNoised(keys=["image"], prob=0.3, mean=0.0, std=0.05),
    EnsureTyped(keys=["image"], dtype=torch.float32)
])

val_transforms = Compose([
    EnsureChannelFirstd(keys=["image"], channel_dim=-1),
    ScaleIntensityRangePercentilesd(
        keys=["image"], lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image"], spatial_size=(224, 224)),
    EnsureTyped(keys=["image"], dtype=torch.float32)
])


def extract_soft_crop(img_np, mask_path, pad_ratio=0.25):
    """
    Loads kidney mask (.mat or image), computes bounding box, and adds a soft
    padding margin to preserve surrounding anatomical context.
    """
    H, W = img_np.shape[:2]

    # Load mask if available
    mask = None
    if os.path.exists(mask_path):
        if mask_path.endswith('.mat'):
            mat = sio.loadmat(mask_path)
            # Adapt key based on your .mat structure (e.g., 'mask' or 'kidney')
            mask_key = [k for k in mat.keys() if not k.startswith('__')][0]
            mask = mat[mask_key]
        else:
            mask = np.array(Image.open(mask_path).convert('L'))

    # Fallback to full image if mask missing or empty
    if mask is None or np.sum(mask > 0) == 0:
        return img_np

    # Find non-zero mask bounding box
    rows = np.any(mask > 0, axis=1)
    cols = np.any(mask > 0, axis=0)
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]

    # Compute bounding box dimensions and soft padding
    h_box = ymax - ymin
    w_box = xmax - xmin
    pad_h = int(h_box * pad_ratio)
    pad_w = int(w_box * pad_ratio)

    # Apply padding within valid image boundaries
    ymin = max(0, ymin - pad_h)
    ymax = min(H, ymax + pad_h)
    xmin = max(0, xmin - pad_w)
    xmax = min(W, xmax + pad_w)

    return img_np[ymin:ymax, xmin:xmax]


class MONAIKidneyDataset(Dataset):
    def __init__(self, image_paths, mask_paths, labels, transform, mode="full"):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.labels = labels
        self.transform = transform
        self.mode = mode

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load raw image
        raw_img = Image.open(self.image_paths[idx]).convert("RGB")
        img_np = np.array(raw_img)

        # Apply soft crop if kidney-focused mode is active
        if self.mode == "kidney_focused" and self.mask_paths is not None:
            img_np = extract_soft_crop(img_np, self.mask_paths[idx], pad_ratio=0.25)

        # Scale intensity to [0, 1] keeping shape (H, W, C)
        img_np = img_np.astype(np.float32) / 255.0

        # MONAI handles channel axis conversion via channel_dim=-1
        data_dict = {"image": img_np}
        data_dict = self.transform(data_dict)

        img_tensor = data_dict["image"]
        if img_tensor.shape[0] == 1:
            img_tensor = img_tensor.repeat(3, 1, 1)

        return {
            "image": img_tensor,
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


def build_efficientnet_model(num_classes=2):
    model = timm.create_model("efficientnet_b0", pretrained=True, in_chans=3)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def evaluate_performance(model, dataloader, device):
    """
    Computes required evaluation standards: ROC-AUC, Sensitivity, Specificity, F1.
    """
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images)
            probs = F.softmax(outputs, dim=1)[:, 1]

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    preds = (all_probs >= 0.5).astype(int)

    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return auc, sensitivity, specificity, f1, all_labels, all_probs


# --- 6. Main 5-Fold Cross-Validation Pipeline ---
def train_and_eval_mode(mode="full", folds_csv_path="/content/drive/MyDrive/DS Project/folds.csv"):
    print(f"5-fold CV for mode: {mode.upper()}")

    df = pd.read_csv(folds_csv_path)
    label_map = {'healthy': 0, 'pathological': 1}
    if df['diagnosis'].dtype == object:
        df['label'] = df['diagnosis'].map(label_map)
    else:
        df['label'] = df['diagnosis'].astype(int)

    IMAGES_DIR = "dataset/images"
    MASKS_DIR = "dataset/masks"
    BATCH_SIZE = 16
    EPOCHS = 10
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Map image and mask paths
    df['image_path'] = df['filename'].apply(lambda x: os.path.join(IMAGES_DIR, x if x.endswith('.jpg') else f"{x}.jpg"))
    df['mask_path'] = df['filename'].apply(lambda x: os.path.join(MASKS_DIR, x.replace('.jpg', '.mat')))

    fold_metrics = []

    for fold in range(5):
        print(f"\n--- Fold {fold+1}/5 ---")
        train_df = df[df['fold'] != fold].reset_index(drop=True)
        val_df = df[df['fold'] == fold].reset_index(drop=True)

        train_ds = MONAIKidneyDataset(
            train_df['image_path'].values, train_df['mask_path'].values,
            train_df['label'].values, transform=train_transforms, mode=mode
        )
        val_ds = MONAIKidneyDataset(
            val_df['image_path'].values, val_df['mask_path'].values,
            val_df['label'].values, transform=val_transforms, mode=mode
        )

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = build_efficientnet_model(num_classes=2).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=6, gamma=0.5)
        criterion = nn.CrossEntropyLoss()

        best_val_auc = 0.0
        best_metrics = None

        for epoch in range(EPOCHS):
            model.train()
            running_loss = 0.0

            for batch in train_loader:
                images = batch["image"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)

            scheduler.step()
            val_auc, val_sens, val_spec, val_f1, _, _ = evaluate_performance(model, val_loader, DEVICE)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_metrics = (val_auc, val_sens, val_spec, val_f1)
                torch.save(model.state_dict(), f"best_efficientnet_{mode}_fold{fold}.pth")

            epoch_loss = running_loss / len(train_ds)
            print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Loss: {epoch_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")

        print(f"Fold {fold+1} Best Metrics -> AUC: {best_metrics[0]:.4f} | Sens: {best_metrics[1]:.4f} | Spec: {best_metrics[2]:.4f} | F1: {best_metrics[3]:.4f}")
        fold_metrics.append(best_metrics)

    # Average metrics over all 5 folds
    avg_metrics = np.mean(fold_metrics, axis=0)
    print(f"\nAverage 5-fold preformance ({mode.upper()})")
    print(f"ROC-AUC: {avg_metrics[0]:.4f} | Sensitivity: {avg_metrics[1]:.4f} | Specificity: {avg_metrics[2]:.4f} | F1: {avg_metrics[3]:.4f}")

    return avg_metrics

In [6]:
full_results = train_and_eval_mode(mode="full")

# 2. Run Kidney-Focused (Soft-Padded Crop) Experiment[cite: 3]

# 3. Print Final Comparative Summary Table
print(f"{'Input Mode':<20} | {'ROC-AUC':<8} | {'Sens.':<8} | {'Spec.':<8} | {'F1-Score':<8}")
print(f"{'Full Image':<20} | {full_results[0]:<8.4f} | {full_results[1]:<8.4f} | {full_results[2]:<8.4f} | {full_results[3]:<8.4f}")

5-fold CV for mode: FULL

--- Fold 1/5 ---
Epoch [01/10] - Loss: 0.5293 | Val ROC-AUC: 0.7326
Epoch [02/10] - Loss: 0.4301 | Val ROC-AUC: 0.7219
Epoch [03/10] - Loss: 0.3598 | Val ROC-AUC: 0.7841
Epoch [04/10] - Loss: 0.3231 | Val ROC-AUC: 0.8169
Epoch [05/10] - Loss: 0.2538 | Val ROC-AUC: 0.7798
Epoch [06/10] - Loss: 0.2121 | Val ROC-AUC: 0.7927
Epoch [07/10] - Loss: 0.1572 | Val ROC-AUC: 0.7695
Epoch [08/10] - Loss: 0.1342 | Val ROC-AUC: 0.7856
Epoch [09/10] - Loss: 0.0948 | Val ROC-AUC: 0.7755
Epoch [10/10] - Loss: 0.0968 | Val ROC-AUC: 0.7717
Fold 1 Best Metrics -> AUC: 0.8169 | Sens: 0.7720 | Spec: 0.6556 | F1: 0.8243

--- Fold 2/5 ---
Epoch [01/10] - Loss: 0.5345 | Val ROC-AUC: 0.7519
Epoch [02/10] - Loss: 0.4322 | Val ROC-AUC: 0.8107
Epoch [03/10] - Loss: 0.3717 | Val ROC-AUC: 0.7983
Epoch [04/10] - Loss: 0.3389 | Val ROC-AUC: 0.8434
Epoch [05/10] - Loss: 0.2662 | Val ROC-AUC: 0.8527
Epoch [06/10] - Loss: 0.2250 | Val ROC-AUC: 0.8324
Epoch [07/10] - Loss: 0.1748 | Val ROC-AUC: 0

Epoch [10/10] - Loss: 0.1249 | Val ROC-AUC: 0.8515
Fold 2 Best Metrics -> AUC: 0.8527 | Sens: 0.8567 | Spec: 0.6222 | F1: 0.8709

--- Fold 3/5 ---
Epoch [01/10] - Loss: 0.5366 | Val ROC-AUC: 0.7647
Epoch [02/10] - Loss: 0.4410 | Val ROC-AUC: 0.8053
Epoch [03/10] - Loss: 0.3835 | Val ROC-AUC: 0.8215
Epoch [04/10] - Loss: 0.3242 | Val ROC-AUC: 0.8004
Epoch [05/10] - Loss: 0.2912 | Val ROC-AUC: 0.8045
Epoch [06/10] - Loss: 0.2247 | Val ROC-AUC: 0.7864
Epoch [07/10] - Loss: 0.1641 | Val ROC-AUC: 0.7890
Epoch [08/10] - Loss: 0.1467 | Val ROC-AUC: 0.7910
Epoch [09/10] - Loss: 0.1166 | Val ROC-AUC: 0.7944
Epoch [10/10] - Loss: 0.1084 | Val ROC-AUC: 0.7733
Fold 3 Best Metrics -> AUC: 0.8215 | Sens: 0.9510 | Spec: 0.3556 | F1: 0.8885

--- Fold 4/5 ---
Epoch [01/10] - Loss: 0.5440 | Val ROC-AUC: 0.7614
Epoch [02/10] - Loss: 0.4360 | Val ROC-AUC: 0.8153
Epoch [03/10] - Loss: 0.3865 | Val ROC-AUC: 0.8411
Epoch [04/10] - Loss: 0.3369 | Val ROC-AUC: 0.8393
Epoch [05/10] - Loss: 0.2725 | Val ROC-AUC:

In [6]:
crop_results = train_and_eval_mode(mode="kidney_focused")

# 3. Print Final Comparative Summary Table
print(f"{'Input Mode':<20} | {'ROC-AUC':<8} | {'Sens.':<8} | {'Spec.':<8} | {'F1-Score':<8}")
print(f"{'Kidney-Focused':<20} | {crop_results[0]:<8.4f} | {crop_results[1]:<8.4f} | {crop_results[2]:<8.4f} | {crop_results[3]:<8.4f}")


5-fold CV for mode: KIDNEY_FOCUSED

--- Fold 1/5 ---


Epoch [01/10] - Loss: 0.5391 | Val ROC-AUC: 0.7786
Epoch [02/10] - Loss: 0.4184 | Val ROC-AUC: 0.8156
Epoch [03/10] - Loss: 0.3554 | Val ROC-AUC: 0.8193
Epoch [04/10] - Loss: 0.2866 | Val ROC-AUC: 0.8380
Epoch [05/10] - Loss: 0.2398 | Val ROC-AUC: 0.8341
Epoch [06/10] - Loss: 0.1860 | Val ROC-AUC: 0.8203
Epoch [07/10] - Loss: 0.1607 | Val ROC-AUC: 0.8352
Epoch [08/10] - Loss: 0.1426 | Val ROC-AUC: 0.8307
Epoch [09/10] - Loss: 0.1105 | Val ROC-AUC: 0.8281
Epoch [10/10] - Loss: 0.1026 | Val ROC-AUC: 0.8378
Fold 1 Best Metrics -> AUC: 0.8380 | Sens: 0.8306 | Spec: 0.6333 | F1: 0.8571

--- Fold 2/5 ---
Epoch [01/10] - Loss: 0.5421 | Val ROC-AUC: 0.7840
Epoch [02/10] - Loss: 0.4241 | Val ROC-AUC: 0.8320
Epoch [03/10] - Loss: 0.3595 | Val ROC-AUC: 0.8754
Epoch [04/10] - Loss: 0.3103 | Val ROC-AUC: 0.8807
Epoch [05/10] - Loss: 0.2484 | Val ROC-AUC: 0.8653
Epoch [06/10] - Loss: 0.1982 | Val ROC-AUC: 0.8647
Epoch [07/10] - Loss: 0.1545 | Val ROC-AUC: 0.8766
Epoch [08/10] - Loss: 0.1255 | Val RO